In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/mandi_project_data/"
output_path = base_path + "processed/"

# Load processed datasets
mandi_df = spark.read.parquet(
    output_path + "monthly_mandi_prices"
)

arrivals_df = spark.read.parquet(
    output_path + "monthly_wheat_arrivals"
)

weather_df = spark.read.parquet(
    output_path + "monthly_weather"
)

combined_df = spark.read.parquet(
    output_path + "monthly_combined_intelligence"
)

print("All analytical datasets loaded successfully!")

All analytical datasets loaded successfully!


In [0]:
monthly_price_trend = (
    mandi_df
    .groupBy("Month")
    .agg(
        F.avg("Average_Modal_Price").alias("Average_Wheat_Price"),
        F.count("*").alias("District_Count")
    )
    .orderBy("Month")
)

monthly_price_trend.show(12, truncate=False)

+-------+-------------------+--------------+
|Month  |Average_Wheat_Price|District_Count|
+-------+-------------------+--------------+
|2025-01|2902.0445074953836 |246           |
|2025-02|2877.309204366295  |246           |
|2025-03|2679.0328773623764 |254           |
|2025-04|2528.4612791686513 |298           |
|2025-05|2519.6391388139973 |291           |
|2025-06|2535.617021556086  |254           |
|2025-07|2543.4194394429064 |252           |
|2025-08|2598.96922801112   |252           |
|2025-09|2578.0833702080004 |254           |
|2025-10|2566.336096406829  |235           |
|2025-11|2553.3334715378123 |195           |
|2025-12|2554.5044362502176 |218           |
+-------+-------------------+--------------+



In [0]:
display(
    monthly_price_trend.select(
        "Month",
        "Average_Wheat_Price"
    )
)

Month,Average_Wheat_Price
2025-01,2902.0445074953836
2025-02,2877.309204366295
2025-03,2679.0328773623764
2025-04,2528.4612791686513
2025-05,2519.6391388139973
2025-06,2535.617021556086
2025-07,2543.4194394429064
2025-08,2598.96922801112
2025-09,2578.0833702080004
2025-10,2566.336096406829


In [0]:
monthly_arrivals_trend = (
    arrivals_df
    .groupBy("Month")
    .agg(
        F.sum("Total_Arrival_Quantity").alias("Total_Wheat_Arrivals"),
        F.count("*").alias("District_Count")
    )
    .orderBy("Month")
)

monthly_arrivals_trend.show(12, truncate=False)

+-------+--------------------+--------------+
|Month  |Total_Wheat_Arrivals|District_Count|
+-------+--------------------+--------------+
|2025-01|818951.5199999998   |229           |
|2025-02|849357.06           |225           |
|2025-03|2845866.9300000016  |231           |
|2025-04|1.434329076E7       |273           |
|2025-05|4984013.979999999   |266           |
|2025-06|1951353.06          |233           |
|2025-07|1421109.6600000001  |231           |
|2025-08|1247637.8599999996  |230           |
|2025-09|1366050.0699999994  |231           |
|2025-10|971151.77           |221           |
+-------+--------------------+--------------+



In [0]:
display(
    monthly_arrivals_trend.select(
        "Month",
        "Total_Wheat_Arrivals"
    )
)


Month,Total_Wheat_Arrivals
2025-01,818951.5199999998
2025-02,849357.06
2025-03,2845866.9300000016
2025-04,1.434329076E7
2025-05,4984013.979999999
2025-06,1951353.06
2025-07,1421109.6600000001
2025-08,1247637.8599999996
2025-09,1366050.0699999994
2025-10,971151.77


In [0]:
price_arrival_comparison = (
    monthly_price_trend
    .join(
        monthly_arrivals_trend,
        on="Month",
        how="inner"
    )
    .orderBy("Month")
)

price_arrival_comparison.show(12, truncate=False)

+-------+-------------------+--------------+--------------------+--------------+
|Month  |Average_Wheat_Price|District_Count|Total_Wheat_Arrivals|District_Count|
+-------+-------------------+--------------+--------------------+--------------+
|2025-01|2902.0445074953836 |246           |818951.5199999998   |229           |
|2025-02|2877.309204366295  |246           |849357.06           |225           |
|2025-03|2679.0328773623764 |254           |2845866.9300000016  |231           |
|2025-04|2528.4612791686513 |298           |1.434329076E7       |273           |
|2025-05|2519.6391388139973 |291           |4984013.979999999   |266           |
|2025-06|2535.617021556086  |254           |1951353.06          |233           |
|2025-07|2543.4194394429064 |252           |1421109.6600000001  |231           |
|2025-08|2598.96922801112   |252           |1247637.8599999996  |230           |
|2025-09|2578.0833702080004 |254           |1366050.0699999994  |231           |
|2025-10|2566.336096406829  

In [0]:
correlation = price_arrival_comparison.select(
    F.corr(
        "Average_Wheat_Price",
        "Total_Wheat_Arrivals"
    ).alias("Price_Arrivals_Correlation")
)

correlation.show(truncate=False)

+--------------------------+
|Price_Arrivals_Correlation|
+--------------------------+
|-0.3742577325457114       |
+--------------------------+



In [0]:
top_price_districts = (
    mandi_df
    .groupBy("District")
    .agg(
        F.avg("Average_Modal_Price").alias("Average_Price")
    )
    .orderBy(F.desc("Average_Price"))
)

top_price_districts.show(10, truncate=False)

+-----------+------------------+
|District   |Average_Price     |
+-----------+------------------+
|Narayanpur |4500.0            |
|Mumbai     |4327.845065562457 |
|Bastar     |4236.0            |
|Bangalore  |3954.522727272727 |
|Shimoga    |3857.1175619990395|
|Chitradurga|3838.0            |
|Kolhapur   |3768.75           |
|Sangli     |3694.784896079689 |
|Mysore     |3637.7698412698414|
|Durg       |3482.0            |
+-----------+------------------+
only showing top 10 rows


In [0]:
bottom_price_districts = (
    mandi_df
    .groupBy("District")
    .agg(
        F.avg("Average_Modal_Price").alias("Average_Price")
    )
    .orderBy(F.asc("Average_Price"))
)

bottom_price_districts.show(10, truncate=False)

+---------------+------------------+
|District       |Average_Price     |
+---------------+------------------+
|Jashpur        |2096.5151515151515|
|Adilabad       |2098.7395833333335|
|Chikmagalur    |2243.4523809523807|
|Faridabad      |2280.0            |
|Kabirdham      |2289.056537436578 |
|Balaghat       |2323.716081126308 |
|Devbhumi Dwarka|2342.754295929457 |
|Janjgir        |2350.0            |
|Palwal         |2350.021505376344 |
|Surajpur       |2353.9027777777774|
+---------------+------------------+
only showing top 10 rows


In [0]:
state_price_analysis = (
    mandi_df
    .groupBy("State")
    .agg(
        F.avg("Average_Modal_Price").alias("Average_Wheat_Price"),
        F.count("*").alias("Record_Count"),
        F.countDistinct("District").alias("District_Count")
    )
    .orderBy(F.desc("Average_Wheat_Price"))
)

state_price_analysis.show(10, truncate=False)

+--------------+-------------------+------------+--------------+
|State         |Average_Wheat_Price|Record_Count|District_Count|
+--------------+-------------------+------------+--------------+
|Karnataka     |3105.9757081583866 |98          |17            |
|Nct Of Delhi  |2799.0654493835864 |12          |1             |
|West Bengal   |2786.415889684381  |73          |7             |
|Maharashtra   |2769.862172694883  |324         |29            |
|Bihar         |2724.18189268461   |20          |5             |
|Gujarat       |2604.007370821084  |288         |25            |
|Rajasthan     |2603.0393264364525 |407         |39            |
|Uttar Pradesh |2589.021310546216  |854         |73            |
|Madhya Pradesh|2566.5441504289    |600         |51            |
|Uttarakhand   |2497.5             |1           |1             |
+--------------+-------------------+------------+--------------+
only showing top 10 rows


In [0]:
state_price_analysis.orderBy(
    F.asc("Average_Wheat_Price")
).show(10, truncate=False)

+--------------+-------------------+------------+--------------+
|State         |Average_Wheat_Price|Record_Count|District_Count|
+--------------+-------------------+------------+--------------+
|Telangana     |2408.133269671505  |17          |2             |
|Haryana       |2441.0527727334866 |55          |21            |
|Uttrakhand    |2441.5897462895    |23          |4             |
|Punjab        |2482.049801344354  |62          |21            |
|Chattisgarh   |2485.364909137052  |161         |23            |
|Uttarakhand   |2497.5             |1           |1             |
|Madhya Pradesh|2566.5441504289    |600         |51            |
|Uttar Pradesh |2589.021310546216  |854         |73            |
|Rajasthan     |2603.0393264364525 |407         |39            |
|Gujarat       |2604.007370821084  |288         |25            |
+--------------+-------------------+------------+--------------+
only showing top 10 rows


In [0]:
mandi_df = (
    mandi_df
    .withColumn(
        "State",
        F.when(F.col("State") == "Uttrakhand", "Uttarakhand")
         .when(F.col("State") == "Chattisgarh", "Chhattisgarh")
         .otherwise(F.col("State"))
    )
)


In [0]:
state_price_analysis = (
    mandi_df
    .groupBy("State")
    .agg(
        F.avg("Average_Modal_Price").alias("Average_Wheat_Price"),
        F.count("*").alias("Record_Count"),
        F.countDistinct("District").alias("District_Count")
    )
    .orderBy(F.desc("Average_Wheat_Price"))
)

state_price_analysis.show(30, truncate=False)


+--------------+-------------------+------------+--------------+
|State         |Average_Wheat_Price|Record_Count|District_Count|
+--------------+-------------------+------------+--------------+
|Karnataka     |3105.9757081583866 |98          |17            |
|Nct Of Delhi  |2799.0654493835864 |12          |1             |
|West Bengal   |2786.415889684381  |73          |7             |
|Maharashtra   |2769.862172694883  |324         |29            |
|Bihar         |2724.18189268461   |20          |5             |
|Gujarat       |2604.007370821084  |288         |25            |
|Rajasthan     |2603.0393264364525 |407         |39            |
|Uttar Pradesh |2589.021310546216  |854         |73            |
|Madhya Pradesh|2566.5441504289    |600         |51            |
|Chhattisgarh  |2485.3649091370526 |161         |23            |
|Punjab        |2482.049801344354  |62          |21            |
|Uttarakhand   |2443.9193401941043 |24          |4             |
|Haryana       |2441.0527

In [0]:
district_price_volatility = (
    mandi_df
    .groupBy("State", "District")
    .agg(
        F.stddev("Average_Modal_Price").alias("Price_Volatility"),
        F.avg("Average_Modal_Price").alias("Average_Price"),
        F.count("*").alias("Month_Count")
    )
    .filter(F.col("Month_Count") >= 6)
    .orderBy(F.desc("Price_Volatility"))
)

district_price_volatility.show(10, truncate=False)

+------------+-----------+------------------+------------------+-----------+
|State       |District   |Price_Volatility  |Average_Price     |Month_Count|
+------------+-----------+------------------+------------------+-----------+
|Karnataka   |Hassan     |551.1422320242161 |3308.958333333333 |8          |
|Telangana   |Medak      |516.4209679810718 |2683.14987974988  |9          |
|Maharashtra |Pune       |434.41479195949586|2983.3239466657265|12         |
|Chhattisgarh|Raipur     |426.82523479054925|2498.793270676692 |10         |
|Karnataka   |Chikmagalur|378.9442047550805 |2243.4523809523807|7          |
|Karnataka   |Dharwad    |364.6264671039747 |2861.757621671258 |11         |
|Karnataka   |Davangere  |331.4681663179541 |2450.440476190476 |10         |
|Karnataka   |Belgaum    |300.73333819541944|3156.9675925925926|9          |
|Telangana   |Adilabad   |287.0746454470191 |2098.7395833333335|8          |
|Punjab      |Mohali     |262.330910912985  |2664.739583333333 |8          |

In [0]:
district_volatility = (
    mandi_df
    .groupBy("State", "District")
    .agg(
        F.stddev("Average_Modal_Price").alias("Price_Volatility"),
        F.avg("Average_Modal_Price").alias("Average_Price"),
        F.countDistinct("Month").alias("Month_Count")
    )
    .filter(F.col("Month_Count") >= 6)
)

In [0]:
district_volatility.orderBy(
    F.asc("Price_Volatility")
).show(10, truncate=False)

+--------------+-----------+-----------------+------------------+-----------+
|State         |District   |Price_Volatility |Average_Price     |Month_Count|
+--------------+-----------+-----------------+------------------+-----------+
|Chhattisgarh  |Jashpur    |21.33514110797807|2096.5151515151515|12         |
|Rajasthan     |Bikaner    |45.33562863768238|2602.5982826050995|11         |
|Gujarat       |Bharuch    |62.11962363504745|2979.9638939316355|11         |
|Chhattisgarh  |Surguja    |64.41696512554438|2383.504785113876 |11         |
|Uttar Pradesh |Maharajganj|65.22232232990227|2511.027293588871 |12         |
|Uttarakhand   |Haridwar   |65.64738190463406|2456.933546789179 |11         |
|Rajasthan     |Churu      |67.46974630652366|2488.8333333333335|6          |
|Madhya Pradesh|Sidhi      |69.44765315289854|2470.216921158292 |12         |
|Chhattisgarh  |Surajpur   |71.71808959390923|2353.9027777777774|9          |
|Maharashtra   |Bhandara   |73.64853577065519|2482.911936845469 

In [0]:
(
    mandi_df
    .filter(
        (F.col("District") == "Hassan") &
        (F.col("State") == "Karnataka")
    )
    .select(
        "Month",
        "Average_Min_Price",
        "Average_Max_Price",
        "Average_Modal_Price"
    )
    .orderBy("Month")
    .show(15, truncate=False)
)


+-------+------------------+------------------+-------------------+
|Month  |Average_Min_Price |Average_Max_Price |Average_Modal_Price|
+-------+------------------+------------------+-------------------+
|2025-01|3533.6666666666665|3753.3333333333335|3533.6666666666665 |
|2025-02|3850.0            |3850.0            |3850.0             |
|2025-03|3350.0            |3725.0            |3350.0             |
|2025-04|2768.0            |3182.5            |2968.0             |
|2025-06|3425.0            |4175.0            |3650.0             |
|2025-07|3416.6666666666665|3866.6666666666665|3566.6666666666665 |
|2025-08|3303.3333333333335|3536.6666666666665|3453.3333333333335 |
|2025-09|2100.0            |2300.0            |2100.0             |
+-------+------------------+------------------+-------------------+



In [0]:
(
    mandi_df
    .filter(
        (F.col("District") == "Jashpur") &
        (F.col("State") == "Chhattisgarh")
    )
    .select(
        "Month",
        "Average_Modal_Price"
    )
    .orderBy("Month")
    .show(15, truncate=False)
)

+-------+-------------------+
|Month  |Average_Modal_Price|
+-------+-------------------+
|2025-01|2100.0             |
|2025-02|2033.3333333333333 |
|2025-03|2116.6666666666665 |
|2025-04|2100.0             |
|2025-05|2090.0             |
|2025-06|2100.0             |
|2025-07|2100.0             |
|2025-08|2100.0             |
|2025-09|2118.181818181818  |
|2025-10|2100.0             |
|2025-11|2100.0             |
|2025-12|2100.0             |
+-------+-------------------+



In [0]:
mandi_df.filter(
    F.col("Average_Modal_Price") <= 0
).count()

0

In [0]:
from pyspark.sql import functions as F

mandi_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in mandi_df.columns
]).show(truncate=False)

+-----+--------+-----+-----------------+-----------------+-------------------+------------------+
|State|District|Month|Average_Min_Price|Average_Max_Price|Average_Modal_Price|Price_Record_Count|
+-----+--------+-----+-----------------+-----------------+-------------------+------------------+
|0    |0       |0    |1                |2                |0                  |0                 |
+-----+--------+-----+-----------------+-----------------+-------------------+------------------+



In [0]:
duplicate_count = (
    mandi_df.count() -
    mandi_df.dropDuplicates(
        ["State", "District", "Month"]
    ).count()
)

print("Duplicate records:", duplicate_count)

Duplicate records: 0


In [0]:
arrivals_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in arrivals_df.columns
]).show(truncate=False)

+-----+--------+-----+----------------------+
|State|District|Month|Total_Arrival_Quantity|
+-----+--------+-----+----------------------+
|0    |0       |0    |0                     |
+-----+--------+-----+----------------------+



In [0]:
duplicate_arrivals = (
    arrivals_df.count() -
    arrivals_df.dropDuplicates(
        ["State", "District", "Month"]
    ).count()
)

print("Duplicate arrivals records:", duplicate_arrivals)

Duplicate arrivals records: 0


In [0]:
weather_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in weather_df.columns
]).show(truncate=False)


+-----+--------+-----+-------------------+-----------------------+-----------------------+-------------------+----------------+------------------+
|State|District|Month|Average_Temperature|Average_Max_Temperature|Average_Min_Temperature|Total_Precipitation|Average_Humidity|Average_Wind_Speed|
+-----+--------+-----+-------------------+-----------------------+-----------------------+-------------------+----------------+------------------+
|0    |0       |0    |0                  |0                      |0                      |0                  |0               |0                 |
+-----+--------+-----+-------------------+-----------------------+-----------------------+-------------------+----------------+------------------+



In [0]:
duplicate_weather = (
    weather_df.count() -
    weather_df.dropDuplicates(
        ["State", "District", "Month"]
    ).count()
)

print("Duplicate weather records:", duplicate_weather)

Duplicate weather records: 0
